In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import VGG16_Weights

In [3]:
!pip install pyyaml

In [2]:
from ssd import SSD
from utils import normalised_gt_coords,corner_to_center,iou,center_to_corner,decode,encode ,matching



"""targets (tensor): Ground truth boxes and labels for a batch,
                shape: [batch_size,num_objs,5] (last idx is the label)."""

gt_boxes_batch = [
    # Image 1
    [
        (75, 80, 60, 90),
        (220, 70, 50, 40),
        (150, 170, 140, 110),
        (260, 240, 70, 120),
        (45, 230, 80, 60)
    ],

    # Image 2
    [
        (30, 40, 50, 60),
        (200, 120, 80, 90),
        (180, 160, 100, 110),
        (250, 210, 60, 80),
        (90, 200, 70, 50),
        (90, 200, 75, 50)
    ],
]

labels_list = [
    torch.tensor([1, 2, 1, 1, 2]),
    torch.tensor([1, 3, 3, 4, 2, 2]) 
]

# convert to tensors
gt_boxes_list = [torch.tensor(b, dtype=torch.float32) for b in gt_boxes_batch]

# gt=center_to_corner(normalised_gt_coords(gtboxes,300,300))


In [3]:
gt_list=[center_to_corner(normalised_gt_coords(gtboxes,300,300)) for gtboxes in gt_boxes_list]
gt_list

[tensor([[0.1500, 0.1167, 0.3500, 0.4167],
         [0.6500, 0.1667, 0.8167, 0.3000],
         [0.2667, 0.3833, 0.7333, 0.7500],
         [0.7500, 0.6000, 0.9833, 1.0000],
         [0.0167, 0.6667, 0.2833, 0.8667]]),
 tensor([[0.0167, 0.0333, 0.1833, 0.2333],
         [0.5333, 0.2500, 0.8000, 0.5500],
         [0.4333, 0.3500, 0.7667, 0.7167],
         [0.7333, 0.5667, 0.9333, 0.8333],
         [0.1833, 0.5833, 0.4167, 0.7500],
         [0.1750, 0.5833, 0.4250, 0.7500]])]

In [4]:
vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
vgg[16] = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)
base=nn.ModuleList(vgg[:30])#until 5_3 layer
base




ModuleList(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (17): Conv2d(256, 512, kernel_siz

In [12]:
X=torch.randn((2,3,300,300))

model=SSD(base,21,alpha=0.1)

In [13]:
regressions,classifications=model(X)

In [14]:
classifications.shape

torch.Size([2, 8732, 21])

In [10]:
from multiloss import MultiLoss

In [11]:
m=MultiLoss()
m.forward(0,2,model.anchors,gt_list,labels_list,regressions,classifications)

(tensor(0.9604, grad_fn=<DivBackward0>),
 tensor(15.4805, grad_fn=<DivBackward0>),
 False)

In [261]:
labels_space.shape

torch.Size([2, 8732, 1])

In [262]:
HNM(classifications[0,:,:],labels_space[0,:,:].squeeze(1))

tensor([5479,  302, 5631,  ..., 8591, 8641, 8691])

In [265]:
F.cross_entropy(classifications[0,:,:],labels_space[0,:,:].squeeze(1),reduction="none") 

tensor([2.8834, 2.9954, 3.0398,  ..., 3.0416, 3.0485, 3.0293],
       grad_fn=<NllLossBackward0>)

In [303]:
labels_reshaped=labels_space.permute(1,0,2).squeeze(-1)
classifications_reshaped=classifications.permute(1,2,0)
losses=F.cross_entropy(classifications_reshaped,labels_reshaped,reduction="none")
losses.shape
# negative_indexes=torch.nonzero(labels_reshaped==0,as_tuple=True,)[0]
# positive_indexes=torch.nonzero(labels_reshaped>0,as_tuple=True,)[0]

torch.Size([8732, 2])

In [305]:
labels_space.permute(1,0,2).squeeze(-1)>0

tensor([[False, False],
        [False, False],
        [False, False],
        ...,
        [False, False],
        [False, False],
        [False, False]])

In [ ]:
labels_space.permute(1,0,2).squeeze(-1).shape

In [ ]:
def HNM(classifications_reshaped,labels_reshaped):

    """
    Input: 

    reshaped classifications; shape [N_anhcores(8732),N_classes] 
    reshaped labels; shape [N_anhcores] , one vector containing classes 

    Output:

    all; shape [positive anchors + top negative anchors]

    For more details : https://arxiv.org/pdf/1512.02325, page 6 

    """
    losses=F.cross_entropy(classifications_reshaped,labels_reshaped,reduction="none") 

    negative_indexes=torch.nonzero(labels_reshaped==0,as_tuple=True,)[0]
    positive_indexes=torch.nonzero(labels_reshaped>0,as_tuple=True,)[0]
    nb_positives=positive_indexes.numel()


    _,indx=losses[negative_indexes].sort(descending=True)

    negative_indexes=negative_indexes[indx[:min(nb_positives*4,len(indx))]]
    
    all=torch.cat([negative_indexes, positive_indexes], dim=0)
    return all


# model forward pass 

In [10]:
X=torch.randn((10,3,300,300))

model=SSD(base,21)
regressions,classifications,layers_for_prediction=model.forward(X)

In [11]:
classifications.shape

torch.Size([10, 8732, 21])

In [46]:
regressions.shape



torch.Size([10, 8732, 4])

In [30]:
for el  in layers_for_prediction:
    print(el.shape[2:])



torch.Size([38, 38])
torch.Size([19, 19])
torch.Size([10, 10])
torch.Size([5, 5])
torch.Size([3, 3])
torch.Size([1, 1])
